# Managing Agent State - Saving and Loading Agents

In [5]:
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_core import CancellationToken

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
import os


In [2]:
load_dotenv(override=True)

from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("✅ API Key loaded successfully.")
else:
    print("❌ API Key not found.")

✅ API Key loaded successfully.


In [3]:
model_client =  OpenAIChatCompletionClient(
    base_url="https://openrouter.ai/api/v1",
    model="qwen/qwen-2.5-72b-instruct:free",
    api_key = api_key,
    model_info={
        "family":'deepseek',
        "vision" :True,
        "function_calling":True,
        "json_output": False
    }
)

/Users/lokesh/Autogen/venv/lib/python3.13/site-packages/autogen_ext/models/openai/_openai_client.py:439: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


In [6]:
assistant = AssistantAgent(
    name = 'Assistant',
    model_client = model_client,
    system_message = 'You are a helpful assistant who can answer questions and provide information on a wide range of topics. You are friendly, informative, and concise.'
)

response = await assistant.on_messages(
    [TextMessage(content = 'Can you help me with a travel itinerary of France from May to June? Keep it under 30 words.',
    source = 'user')],
    cancellation_token = CancellationToken(),
)

print(response.chat_message)

id='d0827339-3ece-4a99-ba1b-43f70b363d09' source='Assistant' models_usage=RequestUsage(prompt_tokens=66, completion_tokens=34) metadata={} created_at=datetime.datetime(2025, 7, 30, 13, 20, 23, 753191, tzinfo=datetime.timezone.utc) content='Sure! Start in Paris, explore the Loire Valley, head to the French Riviera, then finish in Provence. Enjoy the spring blooms and festive markets!' type='TextMessage'


In [7]:
agent_state = await assistant.save_state()

In [8]:
print(agent_state)

{'type': 'AssistantAgentState', 'version': '1.0.0', 'llm_context': {'messages': [{'content': 'Can you help me with a travel itinerary of France from May to June? Keep it under 30 words.', 'source': 'user', 'type': 'UserMessage'}, {'content': 'Sure! Start in Paris, explore the Loire Valley, head to the French Riviera, then finish in Provence. Enjoy the spring blooms and festive markets!', 'thought': None, 'source': 'Assistant', 'type': 'AssistantMessage'}]}}


In [9]:
new_assistant_agent = AssistantAgent(
    name = 'Assistant_agent_2',
    model_client = model_client,
    system_message = 'You are a helpful assistant.'
)

In [10]:
await new_assistant_agent.load_state(agent_state)

In [11]:

response = await new_assistant_agent.on_messages(
    [TextMessage(content = 'Can you tell me what was the last discussion about?',
    source = 'user')],
    cancellation_token = CancellationToken(),
)
print(response.chat_message)

id='8f3f2d02-42ff-4231-a9e2-52bf32469187' source='Assistant_agent_2' models_usage=RequestUsage(prompt_tokens=100, completion_tokens=38) metadata={} created_at=datetime.datetime(2025, 7, 30, 13, 21, 46, 307903, tzinfo=datetime.timezone.utc) content='The last discussion was about creating a travel itinerary for France during May to June, focusing on key regions like Paris, the Loire Valley, the French Riviera, and Provence.' type='TextMessage'


In [12]:
response.chat_message.content

'The last discussion was about creating a travel itinerary for France during May to June, focusing on key regions like Paris, the Loire Valley, the French Riviera, and Provence.'